# Analise Exploratoria - Vacinacao (SI-PNI/RNDS)

Exploracao estrutural da base de vacinacao, a partir de uma amostra nao filtrada de janeiro/2026 (`data/vacinacao/processed/vacinacao_jan_2026_sample.csv`, 50 mil doses de todos os imunobiologicos). Objetivo: entender colunas, tipos, qualidade dos dados (nulos, cardinalidade) antes de recortar por doenca/vacina especifica.

Continua em [03-analise-vacinas-geral.ipynb](03-analise-vacinas-geral.ipynb) (panorama de todas as vacinas) e [04-analise-vacina-sarampo.ipynb](04-analise-vacina-sarampo.ipynb) (recorte de sarampo).

## 1. Setup

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Paleta categorica validada (blue, orange, aqua, yellow, magenta, green, violet, red)
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEQUENTIAL_BLUE = "#2a78d6"
TEXT_PRIMARY = "#0b0b0b"
TEXT_SECONDARY = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
SURFACE = "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID,
    "axes.labelcolor": TEXT_SECONDARY,
    "axes.grid": True,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "text.color": TEXT_PRIMARY,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

SAMPLE_PATH = Path("..") / "data" / "vacinacao" / "processed" / "vacinacao_jan_2026_sample.csv"

df = pd.read_csv(SAMPLE_PATH, sep=";", encoding="latin-1", dtype=str)
df.shape

(50261, 56)

## 2. Visao geral das colunas

In [2]:
df.head()

,co_documento,co_paciente,tp_sexo_paciente,co_raca_cor_paciente,no_raca_cor_paciente,co_municipio_paciente,co_pais_paciente,no_municipio_paciente,no_pais_paciente,sg_uf_paciente,...,ds_condicao_maternal,co_tipo_estabelecimento,ds_tipo_estabelecimento,co_natureza_estabelecimento,ds_natureza_estabelecimento,nu_idade_paciente,co_condicao_maternal,no_uf_paciente,no_uf_estabelecimento,dt_deletado_rnds
0,c208de9a-3522-4fd3-9906-d920f0213455-i0b0,693458605a4b8a643959c681a9d8fedc6237b3f5ccf907...,M,01,BRANCA,410390,10,CAMPINA DA LAGOA,BRASIL,PR,...,Nenhuma,02,CENTRO DE SAUDE/UNIDADE BASICA,1,ADMINISTRACAO PUBLICA,0,1,PARANA,PARANA,NaN
1,11d3583b-41e1-4993-b821-9dab7605ea1c-i0b0,41282e9b3e9900a59ea1bcb297e2264cd5dc4cd7454a80...,F,01,BRANCA,330095,10,COMENDADOR LEVY GASPARIAN,BRASIL,RJ,...,Nenhuma,84,CENTRAL DE ABASTECIMENTO,1,ADMINISTRACAO PUBLICA,0,1,RIO DE JANEIRO,RIO DE JANEIRO,NaN
2,a470d992-d754-4e7f-ade1-7b44bd8cde87-i0b0,5a0be42c57f09fe41cc08964668b4478d8d70db62dd11f...,F,04,AMARELA,210850,10,PINDARE-MIRIM,BRASIL,MA,...,Gestante,02,CENTRO DE SAUDE/UNIDADE BASICA,1,ADMINISTRACAO PUBLICA,15,2,MARANHAO,MARANHAO,NaN
3,925ac9d4-5227-478a-aeac-30ba3b73744c-i0b0,dbd673a123fd6af36592473a21ad1a12a78a5d3cace46a...,M,99,SEM INFORMACAO,290210,10,ARACI,BRASIL,BA,...,Nenhuma,02,CENTRO DE SAUDE/UNIDADE BASICA,1,ADMINISTRACAO PUBLICA,0,1,BAHIA,BAHIA,NaN
4,29e20425-f38d-4f40-8839-b79179876ef8-i0b0,6bfe5e3087289986f4d722c17d06fa9519e82ce3d8e0c3...,M,01,BRANCA,355030,10,SAO PAULO,BRASIL,SP,...,Nenhuma,02,CENTRO DE SAUDE/UNIDADE BASICA,1,ADMINISTRACAO PUBLICA,15,1,SAO PAULO,SAO PAULO,NaN


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50261 entries, 0 to 50260
Data columns (total 56 columns):
 #   Column                           Non-Null Count  Dtype
---  ------                           --------------  -----
 0   co_documento                     50261 non-null  str  
 1   co_paciente                      50261 non-null  str  
 2   tp_sexo_paciente                 50261 non-null  str  
 3   co_raca_cor_paciente             50261 non-null  str  
 4   no_raca_cor_paciente             50261 non-null  str  
 5   co_municipio_paciente            49369 non-null  str  
 6   co_pais_paciente                 49370 non-null  str  
 7   no_municipio_paciente            49370 non-null  str  
 8   no_pais_paciente                 49370 non-null  str  
 9   sg_uf_paciente                   49363 non-null  str  
 10  nu_cep_paciente                  46189 non-null  str  
 11  ds_nacionalidade_paciente        50253 non-null  str  
 12  no_etnia_indigena_paciente       299 non-null    str  
 1

In [4]:
# % de valores ausentes por coluna (todas as colunas, ordenadas)
resumo_nulos = (df.isna().mean() * 100).sort_values(ascending=False)
resumo_nulos

dt_deletado_rnds                   100.000000
no_etnia_indigena_paciente          99.405105
co_etnia_indigena_paciente          99.405105
ds_origem_registro                  96.665407
co_origem_registro                  96.536082
co_troca_documento                  94.070950
nu_cep_paciente                      8.101709
ds_vacina_fabricante                 6.525935
ds_categoria                         5.053620
no_grupo_atendimento                 5.053620
co_categoria                         5.053620
co_vacina_fabricante                 3.292812
co_condicao_maternal                 2.339786
ds_condicao_maternal                 2.339786
co_vacina_grupo_atendimento          2.238316
sg_uf_paciente                       1.786674
no_uf_paciente                       1.774736
co_municipio_paciente                1.774736
no_municipio_paciente                1.772746
co_pais_paciente                     1.772746
no_pais_paciente                     1.772746
no_estrategia                     

In [5]:
# Colunas praticamente vazias (>90% nulo) - candidatas a descarte antes da analise principal
colunas_quase_vazias = resumo_nulos[resumo_nulos > 90]
colunas_quase_vazias

dt_deletado_rnds              100.000000
no_etnia_indigena_paciente     99.405105
co_etnia_indigena_paciente     99.405105
ds_origem_registro             96.665407
co_origem_registro             96.536082
co_troca_documento             94.070950
dtype: float64

In [6]:
# N de valores distintos por coluna - ajuda a distinguir categoricas de identificadores/texto livre
df.nunique().sort_values(ascending=False)

co_documento                       50261
co_paciente                        50083
co_cnes_estabelecimento            20473
no_fantasia_estalecimento          19462
nu_cep_paciente                    16796
no_razao_social_estabelecimento     7342
co_lote_vacina                      4534
co_municipio_paciente               4517
co_municipio_estabelecimento        4451
no_municipio_paciente               4319
no_municipio_estabelecimento        4257
co_sistema_origem                   3585
co_troca_documento                  2980
ds_sistema_origem                    303
dt_entrada_rnds                      226
nu_idade_paciente                    100
no_etnia_indigena_paciente            86
co_etnia_indigena_paciente            86
co_vacina_grupo_atendimento           80
no_grupo_atendimento                  79
co_vacina_fabricante                  75
ds_vacina_fabricante                  72
co_vacina                             64
sg_imunobiologico                     64
ds_nome         

## 3. Colunas de identificacao de vacina/doenca

In [7]:
df[["ds_nome", "sg_imunobiologico", "co_vacina", "ds_tipo_dose"]].head(10)

,ds_nome,sg_imunobiologico,co_vacina,ds_tipo_dose
0,"vacina influenza trivalente (fragmentada, inat...",INF3,33,1ª Dose
1,vacina rotavírus humano G1P[8] (atenuada),ROTA,45,1ª Dose
2,"vacina adsorvida difteria, tétano e pertussis ...",dTpa,57,Dose
3,"vacina adsorvida difteria, tétano, pertussis, ...",Penta,42,1ª Dose
4,vacina adsorvida difteria e tétano adulto,dT,25,Reforço
5,vacina hepatite B (recombinante),HB,9,2ª Dose
6,vacina adsorvida difteria e tétano adulto,dT,25,1ª Dose
7,"vacina influenza trivalente (fragmentada, inat...",INF3,33,1ª Dose
8,"vacina dengue 1, 2, 3 e 4 (atenuada)",DNG,104,2ª Dose
9,vacina pneumocócica 10-valente (conjugada),VPC10,26,Reforço
